# 04 — Baseline Model: Logistic Regression

**Owner:** Person A (modelling track).

**Rubric line:** Sound baseline.

We fit TWO logistic regressions:
1. **Leaky benchmark** — includes `duration`. Useful upper bound, but
   undeployable in real life.
2. **Deployable baseline** — drops `duration`. This is the honest
   comparator that 05's improved model has to beat.

The gap between them is the headline cautionary tale of this project.


In [ ]:
# --- Setup --------------------------------------------------------------
# Make `src/` importable regardless of where the notebook is launched from.
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config, data, features, models, metrics, decision, viz

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 4.1 — Load split

In [ ]:
train_df, test_df = data.load_interim()
y_train = train_df[config.TARGET_COL]
y_test = test_df[config.TARGET_COL]


## 4.2 — Leaky benchmark (with `duration`)

In [ ]:
leaky_cols = data.feature_columns(include_leaky=True)
X_train_leaky = train_df[leaky_cols]
X_test_leaky = test_df[leaky_cols]

leaky_pipe = models.make_logreg_pipeline()
leaky_pipe.fit(X_train_leaky, y_train)
leaky_proba = leaky_pipe.predict_proba(X_test_leaky)[:, 1]
leaky_summary = metrics.classification_summary(y_test.values, leaky_proba)
leaky_summary


## 4.3 — Deployable baseline (no `duration`)

In [ ]:
deploy_cols = data.feature_columns(include_leaky=False)
X_train = train_df[deploy_cols]
X_test = test_df[deploy_cols]

logreg_pipe = models.make_logreg_pipeline()
logreg_pipe.fit(X_train, y_train)
logreg_proba = logreg_pipe.predict_proba(X_test)[:, 1]
logreg_summary = metrics.classification_summary(y_test.values, logreg_proba)
logreg_summary


## 4.4 — Side-by-side comparison

In [ ]:
comparison = pd.DataFrame(
    [leaky_summary, logreg_summary],
    index=['Leaky benchmark (with duration)', 'Deployable baseline (no duration)'],
)
comparison.to_csv(config.TABLES_DIR / 'baseline_comparison.csv')
comparison


## 4.5 — Curves and confusion matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
viz.plot_roc(y_test, logreg_proba, 'Logistic baseline', ax=axes[0])
viz.plot_pr(y_test, logreg_proba, 'Logistic baseline', ax=axes[1])
viz.plot_confusion(y_test, (logreg_proba >= 0.5).astype(int), ax=axes[2])
viz.save_fig(fig, '04_baseline_curves')


## 4.6 — Persist baseline predictions for downstream notebooks

In [ ]:
import joblib
joblib.dump(
    {'model': logreg_pipe, 'proba_test': logreg_proba},
    config.MODELS_DIR / 'baseline_logreg.joblib',
)


## 4.7 — Plain-language interpretation

Two or three sentences for the slide deck: *what does the gap between the
leaky and deployable models tell us?* This is the headline methodological
point of the entire project.
